<p style="margin: 5px 0 0 0; color: #666;"><em>Desarrollado con Claude - Anthropic</em></p>

# 5. Limpieza y Preparación de Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler  # LabelEncoder

## Introducción a la Limpieza de Datos

### ¿Qué es?

La **limpieza y preparación de datos** (Data Cleaning / Data Wrangling) es el proceso de detectar y corregir (o eliminar) registros incorrectos, incompletos, duplicados o irrelevantes de un dataset. Es una de las etapas más importantes y que más tiempo consume en cualquier proyecto de análisis de datos, representando entre el **60-80% del tiempo total** de trabajo.

**Etapas principales de la limpieza de datos:**

| Etapa | Descripción | Ejemplo |
|---|---|---|
| **Identificación de faltantes** | Detectar valores nulos o ausentes | `NaN`, celdas vacías |
| **Tratamiento de outliers** | Identificar y manejar valores extremos | Salario de $10M en nómina |
| **Normalización/Estandarización** | Escalar variables a rangos comparables | Edad vs Salario en misma escala |
| **Codificación categórica** | Convertir texto a números para modelos | "Alto" → 3, "Medio" → 2 |
| **Eliminación de duplicados** | Remover registros repetidos | Mismo cliente registrado dos veces |
| **Validación de datos** | Verificar que los datos cumplan reglas | Edad entre 0 y 120 |
| **Transformación de tipos** | Corregir tipos de datos incorrectos | "123" (texto) → 123 (entero) |
| **Feature Engineering** | Crear variables nuevas a partir de existentes | Año y mes desde una fecha |

### ¿Para qué sirve?

La limpieza y preparación de datos sirve para:

- **Garantizar la calidad del análisis**: conclusiones basadas en datos sucios producen resultados erróneos (*garbage in, garbage out*)
- **Preparar datos para modelos de ML**: la mayoría de algoritmos no aceptan nulos, categorías textuales ni escalas dispares
- **Detectar problemas tempranamente**: identificar errores de captura, inconsistencias y sesgos antes de tomar decisiones
- **Reducir ruido en los datos**: eliminar outliers y duplicados que distorsionan estadísticas y visualizaciones
- **Estandarizar formatos**: unificar fechas, monedas, unidades y categorías de distintas fuentes
- **Aumentar la reproducibilidad**: un pipeline de limpieza documentado permite repetir el proceso de forma consistente

### ¿Cómo se usa?

En las secciones siguientes, abordaremos cada etapa de la limpieza de datos con ejemplos prácticos en Python usando pandas, numpy, matplotlib, seaborn y scikit-learn.

## 1. Identificación de Datos Faltantes

### ¿Qué es?

Los **datos faltantes** (missing values) son valores ausentes en un dataset, representados comúnmente como `NaN` (Not a Number) o `None` en Python. Son uno de los problemas más frecuentes en datos reales y pueden originarse por errores de captura, fallos de sensores, campos opcionales no rellenados o problemas en la integración de fuentes.

**Tipos de datos faltantes:**

| Tipo | Descripción | Ejemplo |
|---|---|---|
| **MCAR** (Missing Completely at Random) | La ausencia es completamente aleatoria | Error aleatorio de transmisión |
| **MAR** (Missing at Random) | La ausencia depende de otra variable observada | Personas mayores no reportan ingresos |
| **MNAR** (Missing Not at Random) | La ausencia depende del propio valor faltante | Ingresos altos omitidos intencionalmente |

**Métodos de detección:**

- `isnull()` / `isna()` → Devuelve máscara booleana de valores nulos
- `isnull().sum()` → Conteo de nulos por columna
- `isnull().mean() * 100` → Porcentaje de nulos por columna
- Heatmaps de nulidad → Visualización de patrones de datos faltantes

### ¿Para qué sirve?

Identificar datos faltantes sirve para:

- **Evaluar la calidad inicial del dataset** antes de cualquier análisis o modelado
- **Decidir la estrategia de tratamiento** adecuada: imputación, eliminación o marcado
- **Detectar patrones de nulidad** que revelen problemas sistemáticos en la recolección de datos
- **Cuantificar el impacto** de los faltantes en el tamaño efectivo del dataset
- **Prevenir errores en modelos** ya que muchos algoritmos no aceptan valores nulos
- **Documentar y reportar** la calidad de los datos a equipos y stakeholders

### ¿Cómo se usa?

En el código siguiente, creamos un dataset con valores faltantes introducidos aleatoriamente y aplicamos los métodos de detección: conteo, porcentajes, heatmap de nulidad y conteo de filas afectadas.

In [ ]:
# Crear dataset con datos faltantes
np.random.seed(42)
df = pd.DataFrame(
    {
        "ID": range(1, 101),
        "Edad": np.random.randint(18, 70, 100),
        "Salario": np.random.randint(30000, 120000, 100),
        "Experiencia": np.random.randint(0, 30, 100),
        "Departamento": np.random.choice(["Ventas", "IT", "RRHH", "Marketing"], 100),
    }
)

# Introducir valores faltantes de forma aleatoria
df.loc[np.random.choice(df.index, 15), "Edad"] = np.nan
df.loc[np.random.choice(df.index, 20), "Salario"] = np.nan
df.loc[np.random.choice(df.index, 10), "Experiencia"] = np.nan

print("=" * 60)
print("IDENTIFICACIÓN DE DATOS FALTANTES")
print("=" * 60)

# Resumen de faltantes
print("\n1. Conteo de valores faltantes por columna:")
print(df.isnull().sum())

print("\n2. Porcentaje de valores faltantes:")
porcentaje_faltantes = (df.isnull().sum() / len(df)) * 100
print(porcentaje_faltantes)

print("\n3. Visualización de patrones de faltantes:")
# Heatmap de valores faltantes
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=True, cmap="viridis", yticklabels=False)
plt.title("Mapa de Valores Faltantes (Amarillo = Faltante)", fontweight="bold")
plt.show()

# Matriz de correlación de faltantes
print("\n4. Filas con al menos un valor faltante:")
print(
    f"Total: {df.isnull().any(axis=1).sum()} de {len(df)} filas ({(df.isnull().any(axis=1).sum() / len(df) * 100):.1f}%)"
)

## 2. Tratamiento de Outliers

### ¿Qué es?

Los **outliers** (valores atípicos) son observaciones que se desvían significativamente del resto de los datos. Pueden ser legítimos (un CEO con salario extremadamente alto) o erróneos (un error de captura). Identificarlos y tratarlos es esencial para que las estadísticas descriptivas y los modelos no se distorsionen.

**Métodos principales de detección:**

| Método | Descripción | Criterio |
|---|---|---|
| **IQR** (Rango Intercuartílico) | Basado en cuartiles Q1 y Q3 | Fuera de [Q1 − 1.5×IQR, Q3 + 1.5×IQR] |
| **Z-Score** | Distancia en desviaciones estándar | \|z\| > 3 (más de 3 desviaciones) |
| **Percentiles** | Basado en percentiles extremos | Fuera de [P5, P95] o [P1, P99] |

**Estrategias de tratamiento:**

- **Eliminación** → Remover los registros con outliers (cuando son errores claros)
- **Capping/Winsorización** → Reemplazar por el límite superior o inferior
- **Transformación** → Aplicar log, sqrt u otra función para reducir el rango
- **Imputación** → Reemplazar con mediana u otro valor representativo
- **Mantener** → Si el outlier es legítimo y relevante para el análisis

### ¿Para qué sirve?

Detectar y tratar outliers sirve para:

- **Evitar distorsiones en estadísticas** como media, varianza y correlaciones
- **Mejorar el rendimiento de modelos** sensibles a valores extremos (regresión lineal, KNN)
- **Identificar errores de datos** que deben corregirse en la fuente
- **Visualizar distribuciones reales** sin que los extremos compriman la escala
- **Decidir caso a caso** si un outlier es un error o un dato legítimo y valioso
- **Documentar umbrales de aceptación** para el monitoreo continuo de calidad

### ¿Cómo se usa?

En el código siguiente, introducimos outliers artificiales en los datos de salario y aplicamos los métodos IQR y Z-Score para detectarlos, los visualizamos con boxplot e histograma, y finalmente removemos los valores extremos.

In [ ]:
print("=" * 60)
print("DETECCIÓN Y TRATAMIENTO DE OUTLIERS")
print("=" * 60)

# Crear columna con outliers
df_clean = df.dropna()
df_outliers = df_clean.copy()
df_outliers.loc[1, "Salario"] = 500000  # Outlier extremo
df_outliers.loc[5, "Salario"] = 350000  # Outlier extremo

# Método 1: Rango intercuartílico (IQR)
print("\n1. MÉTODO IQR (Rango Intercuartílico)")
Q1 = df_outliers["Salario"].quantile(0.25)
Q3 = df_outliers["Salario"].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print(f"Q1 (25%): ${Q1:,.2f}")
print(f"Q3 (75%): ${Q3:,.2f}")
print(f"IQR: ${IQR:,.2f}")
print(f"Límite inferior: ${limite_inferior:,.2f}")
print(f"Límite superior: ${limite_superior:,.2f}")

outliers_iqr = df_outliers[
    (df_outliers["Salario"] < limite_inferior)
    | (df_outliers["Salario"] > limite_superior)
]
print(f"\nOutliers detectados: {len(outliers_iqr)}")
print(outliers_iqr[["ID", "Salario"]])

# Método 2: Z-score
print("\n2. MÉTODO Z-SCORE")
z_scores = np.abs(stats.zscore(df_outliers["Salario"].dropna()))
outliers_zscore = df_outliers[z_scores > 3]
print(f"Outliers detectados (|z| > 3): {len(outliers_zscore)}")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
axes[0].boxplot(df_outliers["Salario"].dropna())
axes[0].set_ylabel("Salario ($)")
axes[0].set_title("Box Plot - Detección de Outliers", fontweight="bold")
axes[0].grid(True, alpha=0.3)

# Histograma
axes[1].hist(df_outliers["Salario"].dropna(), bins=30, edgecolor="black")
axes[1].axvline(limite_inferior, color="red", linestyle="--", label="Límites IQR")
axes[1].axvline(limite_superior, color="red", linestyle="--")
axes[1].set_xlabel("Salario ($)")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("Distribución con Outliers", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.show()

# Tratamiento: Remover outliers
df_sin_outliers = df_outliers[
    (df_outliers["Salario"] >= limite_inferior)
    & (df_outliers["Salario"] <= limite_superior)
]
print("\n3. DATOS DESPUÉS DE REMOVER OUTLIERS")
print(f"Filas antes: {len(df_outliers)}")
print(f"Filas después: {len(df_sin_outliers)}")
print(f"Filas removidas: {len(df_outliers) - len(df_sin_outliers)}")

## 3. Normalización y Estandarización

### ¿Qué es?

La **normalización** y la **estandarización** son técnicas de escalado que transforman variables numéricas para que sean comparables entre sí, especialmente cuando tienen rangos o unidades muy diferentes (por ejemplo, Edad: 18-70 vs Salario: 20,000-150,000).

**Métodos principales:**

| Método | Fórmula | Resultado | Cuándo usar |
|---|---|---|---|
| **Estandarización** (StandardScaler) | $z = \frac{x - \mu}{\sigma}$ | Media = 0, Std = 1 | Distribución normal, presencia de outliers |
| **Normalización** (MinMaxScaler) | $x' = \frac{x - x_{min}}{x_{max} - x_{min}}$ | Rango [0, 1] | Redes neuronales, sin outliers extremos |
| **Robust Scaler** | $x' = \frac{x - Q2}{Q3 - Q1}$ | Basado en mediana e IQR | Datos con muchos outliers |

**Diferencia clave:**

- **Estandarización**: No acota los valores a un rango fijo, más robusta ante outliers
- **Normalización**: Acota los valores entre 0 y 1, sensible a outliers extremos

### ¿Para qué sirve?

Normalizar y estandarizar datos sirve para:

- **Hacer comparables variables con escalas diferentes** para que ninguna domine por su magnitud
- **Mejorar la convergencia de algoritmos** como regresión, SVM, KNN y redes neuronales
- **Cumplir requisitos de modelos** que asumen datos centrados o en rangos específicos
- **Facilitar la interpretación** de coeficientes en modelos lineales
- **Acelerar el entrenamiento** de modelos de Machine Learning
- **Elegir el método adecuado** según la distribución de los datos y la presencia de outliers

### ¿Cómo se usa?

En el código siguiente, aplicamos StandardScaler (estandarización) y MinMaxScaler (normalización) a las variables numéricas, comparamos las estadísticas resultantes y visualizamos las distribuciones antes y después de cada transformación.

In [ ]:
print("=" * 60)
print("NORMALIZACIÓN Y ESTANDARIZACIÓN")
print("=" * 60)

# Datos de ejemplo
df_scale = df_sin_outliers[["Edad", "Salario", "Experiencia"]].dropna().copy()

print("\nEstadísticas originales:")
print(df_scale.describe())

# ESTANDARIZACIÓN (Z-score): media=0, std=1
print("\n1. ESTANDARIZACIÓN (StandardScaler)")
scaler_std = StandardScaler()
df_standardized = pd.DataFrame(
    scaler_std.fit_transform(df_scale),
    columns=["Edad_std", "Salario_std", "Experiencia_std"],
)
print(df_standardized.describe())

# NORMALIZACIÓN (Min-Max): rango 0-1
print("\n2. NORMALIZACIÓN (MinMaxScaler)")
scaler_minmax = MinMaxScaler()
df_normalized = pd.DataFrame(
    scaler_minmax.fit_transform(df_scale),
    columns=["Edad_norm", "Salario_norm", "Experiencia_norm"],
)
print(df_normalized.describe())

# Visualización comparativa
fig, axes = plt.subplots(3, 3, figsize=(15, 12))

columnas = ["Edad", "Salario", "Experiencia"]
for i, col in enumerate(columnas):
    # Original
    axes[i, 0].hist(df_scale[col], bins=20, edgecolor="black", alpha=0.7)
    axes[i, 0].set_title(f"{col} - Original", fontweight="bold")
    axes[i, 0].set_ylabel("Frecuencia")

    # Estandarizado
    axes[i, 1].hist(
        df_standardized[f"{col}_std"],
        bins=20,
        edgecolor="black",
        alpha=0.7,
        color="orange",
    )
    axes[i, 1].set_title(f"{col} - Estandarizado", fontweight="bold")

    # Normalizado
    axes[i, 2].hist(
        df_normalized[f"{col}_norm"],
        bins=20,
        edgecolor="black",
        alpha=0.7,
        color="green",
    )
    axes[i, 2].set_title(f"{col} - Normalizado", fontweight="bold")

plt.tight_layout()
plt.show()

print("\n3. CUÁNDO USAR CADA MÉTODO:")
print("\nEstandarización (StandardScaler):")
print("  - Algoritmos sensibles a escala (SVM, KNN, Regresión)")
print("  - Cuando los datos siguen distribución normal")
print("  - Cuando hay outliers (menos sensible que MinMax)")
print("\nNormalización (MinMaxScaler):")
print("  - Redes neuronales")
print("  - Cuando se necesita rango específico [0,1]")
print("  - Cuando no hay outliers extremos")

## 4. Codificación de Variables Categóricas

### ¿Qué es?

La **codificación de variables categóricas** es el proceso de convertir datos textuales (como "Ventas", "IT", "Junior") en representaciones numéricas que los algoritmos de Machine Learning puedan procesar. Es un paso esencial porque la mayoría de los modelos solo trabajan con datos numéricos.

**Métodos principales:**

| Método | Tipo de variable | Descripción | Ejemplo |
|---|---|---|---|
| **Label Encoding** | Ordinal | Asigna un número entero a cada categoría | Junior=0, Mid=1, Senior=2 |
| **One-Hot Encoding** | Nominal | Crea una columna binaria por categoría | Ventas=[1,0,0], IT=[0,1,0] |
| **Ordinal Encoding** | Ordinal | Asigna números respetando un orden definido | Baja=1, Media=2, Alta=3 |
| **Frequency Encoding** | Nominal/Ordinal | Codifica según la frecuencia de aparición | Ventas=0.33, IT=0.33 |

**¿Cuándo usar cada uno?**

- **Label/Ordinal Encoding**: Cuando las categorías tienen un orden natural (nivel educativo, tallas, satisfacción)
- **One-Hot Encoding**: Cuando las categorías no tienen orden (departamento, ciudad, color). Usar `drop_first=True` para evitar multicolinealidad
- **Frequency Encoding**: Cuando hay alta cardinalidad (muchas categorías únicas) y el orden no importa

### ¿Para qué sirve?

Codificar variables categóricas sirve para:

- **Habilitar el uso de modelos de ML** que requieren entradas exclusivamente numéricas
- **Preservar la semántica ordinal** cuando las categorías tienen un orden significativo
- **Evitar sesgos artificiales** que surgen al asignar números arbitrarios a categorías nominales
- **Controlar la dimensionalidad** eligiendo entre métodos que crean más o menos columnas
- **Manejar alta cardinalidad** usando técnicas como Frequency o Target Encoding
- **Mejorar la interpretabilidad** al hacer explícita la representación de cada categoría

### ¿Cómo se usa?

En el código siguiente, aplicamos los cuatro métodos de codificación a un dataset con variables categóricas: Label Encoding para niveles, One-Hot Encoding para departamentos y ciudades, Ordinal Encoding para satisfacción y Frequency Encoding para frecuencia de aparición.

In [ ]:
print("=" * 60)
print("CODIFICACIÓN DE VARIABLES CATEGÓRICAS")
print("=" * 60)

# Crear dataset con variables categóricas
df_cat = pd.DataFrame(
    {
        "Departamento": ["Ventas", "IT", "RRHH", "Marketing", "Ventas", "IT"],
        "Nivel": ["Junior", "Senior", "Mid", "Senior", "Mid", "Junior"],
        "Ciudad": ["Madrid", "Barcelona", "Valencia", "Madrid", "Barcelona", "Madrid"],
        "Satisfaccion": ["Alta", "Baja", "Media", "Alta", "Alta", "Media"],
    }
)

print("\nDataFrame original:")
print(df_cat)

# 1. LABEL ENCODING - Para variables ordinales
print("\n1. LABEL ENCODING (Variables Ordinales)")
nivel_orden = {"Junior": 0, "Mid": 1, "Senior": 2}
df_cat["Nivel_Encoded"] = df_cat["Nivel"].map(nivel_orden)
print(df_cat[["Nivel", "Nivel_Encoded"]])

# 2. ONE-HOT ENCODING - Para variables nominales
print("\n2. ONE-HOT ENCODING (Variables Nominales)")
df_onehot = pd.get_dummies(df_cat["Departamento"], prefix="Dept")
print(df_onehot)

# Aplicar a todo el DataFrame
print("\n3. ONE-HOT ENCODING completo:")
df_encoded = pd.get_dummies(
    df_cat, columns=["Departamento", "Ciudad"], prefix=["Dept", "City"], drop_first=True
)
print(df_encoded)

# 3. ORDINAL ENCODING con orden personalizado
print("\n4. ORDINAL ENCODING (con orden):")
satisfaccion_orden = {"Baja": 1, "Media": 2, "Alta": 3}
df_cat["Satisfaccion_Ord"] = df_cat["Satisfaccion"].map(satisfaccion_orden)
print(df_cat[["Satisfaccion", "Satisfaccion_Ord"]])

# 4. FREQUENCY ENCODING
print("\n5. FREQUENCY ENCODING:")
freq_encoding = df_cat["Departamento"].value_counts(normalize=True)
df_cat["Dept_Frequency"] = df_cat["Departamento"].map(freq_encoding)
print(df_cat[["Departamento", "Dept_Frequency"]])

print("\n" + "=" * 60)
print("RESUMEN DE MÉTODOS")
print("=" * 60)
print("\nLabel Encoding: Variables ordinales (orden importa)")
print("One-Hot Encoding: Variables nominales (sin orden)")
print("Ordinal Encoding: Variables con orden personalizado")
print("Frequency Encoding: Codificar por frecuencia de aparición")

## 5. Eliminación de Duplicados

### ¿Qué es?

Los **duplicados** son registros que aparecen más de una vez en un dataset, ya sea de forma exacta (todas las columnas idénticas) o parcial (solo algunas columnas clave son iguales). Pueden originarse por errores en la carga de datos, integraciones de múltiples fuentes, re-envíos de formularios o fallos en procesos ETL.

**Tipos de duplicados:**

| Tipo | Descripción | Ejemplo |
|---|---|---|
| **Exactos** | Todas las columnas son idénticas | Misma fila copiada dos veces |
| **Parciales** | Solo columnas clave son iguales | Mismo ID con datos ligeramente diferentes |
| **Semánticos** | Representan lo mismo con formato distinto | "J. García" y "Juan García" |

**Métodos en pandas:**

- `duplicated()` → Marca filas duplicadas (booleano)
- `duplicated(subset=['col'])` → Duplicados en columnas específicas
- `duplicated(keep='first'|'last'|False)` → Controla cuál se marca como duplicado
- `drop_duplicates()` → Elimina las filas marcadas como duplicadas

### ¿Para qué sirve?

Eliminar duplicados sirve para:

- **Evitar inflar métricas** como conteos, sumas y promedios con registros repetidos
- **Garantizar integridad referencial** donde cada entidad aparezca una sola vez
- **Reducir el tamaño del dataset** eliminando información redundante
- **Prevenir sesgos en modelos** que darían más peso a registros duplicados
- **Identificar problemas en la fuente** de datos que generan duplicaciones
- **Mantener consistencia** al decidir cuál versión conservar (primera, última o ninguna)

### ¿Cómo se usa?

En el código siguiente, creamos un dataset con duplicados intencionales y aplicamos los métodos de detección (`duplicated`) y eliminación (`drop_duplicates`) tanto por filas completas como por columnas específicas, controlando cuál ocurrencia conservar.

In [ ]:
print("=" * 60)
print("ELIMINACIÓN DE DUPLICADOS")
print("=" * 60)

# Crear dataset con duplicados
df_dup = pd.DataFrame(
    {
        "ID": [1, 2, 3, 4, 2, 5, 1, 6],
        "Nombre": ["Ana", "Luis", "María", "Carlos", "Luis", "Pedro", "Ana", "Sofia"],
        "Edad": [25, 30, 28, 35, 30, 32, 25, 27],
        "Salario": [50000, 60000, 55000, 70000, 60000, 65000, 50000, 58000],
    }
)

print("\nDataFrame con duplicados:")
print(df_dup)

# Identificar duplicados
print("\n1. IDENTIFICAR DUPLICADOS")
print(f"Filas duplicadas: {df_dup.duplicated().sum()}")
print("\nFilas duplicadas (marcadas):")
print(df_dup[df_dup.duplicated(keep=False)])

# Duplicados en columnas específicas
print("\n2. DUPLICADOS POR COLUMNA ESPECÍFICA (ID)")
print(f"IDs duplicados: {df_dup.duplicated(subset=['ID']).sum()}")
print(df_dup[df_dup.duplicated(subset=["ID"], keep=False)].sort_values("ID"))

# Eliminar duplicados
print("\n3. ELIMINAR DUPLICADOS")
df_sin_dup = df_dup.drop_duplicates()
print(f"Filas antes: {len(df_dup)}")
print(f"Filas después: {len(df_sin_dup)}")
print("\nDataFrame sin duplicados:")
print(df_sin_dup)

# Eliminar duplicados en columna específica, manteniendo primera/última ocurrencia
print("\n4. ELIMINAR DUPLICADOS (mantener última ocurrencia)")
df_sin_dup_last = df_dup.drop_duplicates(subset=["ID"], keep="last")
print(df_sin_dup_last)

## 6. Validación de Datos

### ¿Qué es?

La **validación de datos** es el proceso de verificar que los valores de un dataset cumplen con reglas de negocio, formatos esperados y restricciones lógicas. A diferencia de la detección de faltantes o duplicados, la validación se enfoca en que los datos **presentes** sean correctos y coherentes.

**Tipos de validaciones comunes:**

| Validación | Descripción | Ejemplo |
|---|---|---|
| **Rango** | Valores dentro de límites aceptables | Edad entre 18 y 100 |
| **Formato** | Cumple con un patrón específico | Email contiene "@" y "." |
| **Longitud** | Número de caracteres esperado | Teléfono tiene 9 dígitos |
| **Tipo** | El valor es del tipo correcto | Precio es numérico, no texto |
| **Consistencia** | Relación lógica entre campos | Fecha inicio < Fecha fin |
| **Unicidad** | Valores únicos donde se requiere | IDs no repetidos |

**Estrategias ante datos inválidos:**

- **Rechazar** → Eliminar registros inválidos del dataset
- **Corregir** → Aplicar reglas de corrección automática
- **Marcar** → Agregar columnas de validación para revisión manual
- **Reportar** → Generar resumen de problemas para el equipo de datos

### ¿Para qué sirve?

Validar datos sirve para:

- **Detectar errores de captura** como valores negativos donde no deben existir o formatos incorrectos
- **Garantizar la calidad** de los datos antes de alimentar dashboards, reportes o modelos
- **Establecer reglas de negocio** que definan qué datos son aceptables
- **Automatizar controles de calidad** que se ejecuten cada vez que llegan datos nuevos
- **Documentar criterios de aceptación** para que todo el equipo conozca los estándares
- **Prevenir errores downstream** que se propagarían a análisis y decisiones de negocio

### ¿Cómo se usa?

En el código siguiente, creamos un dataset con valores intencionalmente inválidos y aplicamos validaciones de rango (edad, salario), formato (email con regex) y longitud (teléfono), generando un resumen de calidad y filtrando los registros válidos.

In [ ]:
print("=" * 60)
print("VALIDACIÓN DE DATOS")
print("=" * 60)

# Crear dataset con datos inválidos
df_val = pd.DataFrame(
    {
        "Edad": [25, -5, 150, 30, 45],  # -5 y 150 son inválidos
        "Salario": [50000, 60000, -10000, 70000, 55000],  # -10000 es inválido
        "Email": [
            "ana@mail.com",
            "invalido",
            "maria@mail.com",
            "carlos@mail",
            "pedro@mail.com",
        ],
        "Telefono": ["123456789", "12345", "987654321", "555555555", "texto"],
    }
)

print("\nDataFrame con datos potencialmente inválidos:")
print(df_val)

# Validaciones
print("\n1. VALIDAR RANGOS")
edades_invalidas = df_val[(df_val["Edad"] < 18) | (df_val["Edad"] > 100)]
print(f"Edades fuera de rango (18-100): {len(edades_invalidas)}")
print(edades_invalidas)

salarios_invalidos = df_val[df_val["Salario"] < 0]
print(f"\nSalarios negativos: {len(salarios_invalidos)}")
print(salarios_invalidos)

# Validar formato de email (simple)
print("\n2. VALIDAR FORMATO (Email)")
df_val["Email_Valido"] = df_val["Email"].str.contains("@.*\.", regex=True)
print(df_val[["Email", "Email_Valido"]])
print(f"Emails inválidos: {(~df_val['Email_Valido']).sum()}")

# Validar longitud
print("\n3. VALIDAR LONGITUD (Teléfono debe tener 9 dígitos)")
df_val["Telefono_Valido"] = df_val["Telefono"].str.len() == 9
print(df_val[["Telefono", "Telefono_Valido"]])

# Resumen de validación
print("\n4. RESUMEN DE VALIDACIÓN")
print("=" * 60)
print(f"Total de registros: {len(df_val)}")
print(f"Edades inválidas: {len(edades_invalidas)}")
print(f"Salarios inválidos: {len(salarios_invalidos)}")
print(f"Emails inválidos: {(~df_val['Email_Valido']).sum()}")
print(f"Teléfonos inválidos: {(~df_val['Telefono_Valido']).sum()}")

# Limpiar datos inválidos
print("\n5. LIMPIAR DATOS INVÁLIDOS")
df_limpio = df_val[
    (df_val["Edad"] >= 18)
    & (df_val["Edad"] <= 100)
    & (df_val["Salario"] >= 0)
    & (df_val["Email_Valido"])
    & (df_val["Telefono_Valido"])
].copy()

print(f"Registros antes: {len(df_val)}")
print(f"Registros después: {len(df_limpio)}")
print(f"Registros eliminados: {len(df_val) - len(df_limpio)}")

## 7. Transformación de Tipos de Datos

### ¿Qué es?

La **transformación de tipos** (type casting) es el proceso de convertir columnas de un tipo de dato a otro más apropiado. Es común que al importar datos desde archivos CSV, bases de datos o APIs, las columnas lleguen como texto (`object`) cuando deberían ser numéricas, fechas o booleanas.

**Conversiones más comunes:**

| De | A | Método en pandas | Ejemplo |
|---|---|---|---|
| Texto → Numérico | `int64` / `float64` | `pd.to_numeric()` | "1200" → 1200 |
| Texto → Fecha | `datetime64` | `pd.to_datetime()` | "2024-01-15" → Timestamp |
| Texto → Booleano | `bool` | `.map()` / `.astype()` | "True" → True |
| Texto → Categoría | `category` | `.astype('category')` | Reduce uso de memoria |

**Manejo de errores en conversión:**

- `errors='raise'` → Lanza error si falla (por defecto)
- `errors='coerce'` → Convierte valores inválidos a `NaN`
- `errors='ignore'` → Devuelve el dato original sin convertir

### ¿Para qué sirve?

Transformar tipos de datos sirve para:

- **Habilitar operaciones matemáticas** que no funcionan con cadenas de texto
- **Reducir consumo de memoria** usando tipos más eficientes como `category` o `int32`
- **Permitir comparaciones y ordenamientos** correctos (las fechas como texto no se ordenan bien)
- **Extraer componentes de fechas** como año, mes, día de la semana, que requieren tipo `datetime`
- **Manejar errores de conversión** de forma controlada con `errors='coerce'`
- **Preparar datos para visualización** y análisis donde el tipo correcto es indispensable

### ¿Cómo se usa?

En el código siguiente, creamos un dataset con tipos incorrectos (IDs y precios como texto, fechas como string, booleanos como texto) y los convertimos a sus tipos apropiados, mostrando también cómo manejar errores de conversión con `errors='coerce'`.

In [ ]:
print("=" * 60)
print("TRANSFORMACIÓN DE TIPOS DE DATOS")
print("=" * 60)

# Crear dataset con tipos incorrectos
df_tipos = pd.DataFrame(
    {
        "ID": ["1", "2", "3", "4", "5"],  # Debería ser int
        "Precio": [
            "100.50",
            "200.75",
            "150.25",
            "300.00",
            "250.50",
        ],  # Debería ser float
        "Cantidad": [10, 20, 15, 25, 30],  # Ya es int
        "Fecha": ["2024-01-01", "2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05"],
        "Activo": ["True", "False", "True", "True", "False"],  # Debería ser bool
    }
)

print("\nTipos de datos originales:")
print(df_tipos.dtypes)

# Transformaciones
print("\n1. CONVERTIR A NUMÉRICO")
df_tipos["ID"] = pd.to_numeric(df_tipos["ID"])
df_tipos["Precio"] = pd.to_numeric(df_tipos["Precio"])
print("✓ ID y Precio convertidos a numérico")

print("\n2. CONVERTIR A FECHA")
df_tipos["Fecha"] = pd.to_datetime(df_tipos["Fecha"])
print("✓ Fecha convertida a datetime")

print("\n3. CONVERTIR A BOOLEANO")
df_tipos["Activo"] = df_tipos["Activo"].map({"True": True, "False": False})
print("✓ Activo convertido a booleano")

print("\nTipos de datos después de transformación:")
print(df_tipos.dtypes)

print("\nDataFrame transformado:")
print(df_tipos)
print(df_tipos.info())

# Manejo de errores en conversión
print("\n4. MANEJO DE ERRORES EN CONVERSIÓN")
datos_con_errores = pd.Series(["1", "2", "tres", "4", "5"])
print(f"\nDatos originales: {datos_con_errores.tolist()}")

# Conversión con coercion (valores inválidos -> NaN)
convertido = pd.to_numeric(datos_con_errores, errors="coerce")
print(f"Convertido (errors='coerce'): {convertido.tolist()}")
print(f"Valores NaN generados: {convertido.isna().sum()}")

## 8. Feature Engineering Básico

### ¿Qué es?

El **Feature Engineering** (ingeniería de características) es el proceso de crear nuevas variables a partir de las existentes para mejorar la capacidad predictiva de un modelo o enriquecer el análisis. Es considerado uno de los factores que más impacto tiene en el rendimiento de los modelos de Machine Learning.

**Técnicas principales:**

| Técnica | Descripción | Ejemplo |
|---|---|---|
| **Variables derivadas** | Operaciones entre columnas existentes | Margen = Precio − Costo |
| **Binning** | Agrupar valores continuos en categorías | Precio: Bajo, Medio, Alto |
| **Features de fechas** | Extraer componentes temporales | Mes, día de semana, es fin de semana |
| **Interacciones** | Combinar variables multiplicativamente | Precio × Cantidad |
| **Transformaciones matemáticas** | Aplicar funciones para cambiar distribución | log(Precio), √Ventas |
| **Agregaciones** | Estadísticas por grupo | Promedio de ventas por categoría |

**Buenas prácticas:**

- Comenzar con variables que tengan sentido de negocio
- Validar que las nuevas features aportan información útil
- Documentar el significado de cada variable creada
- Evitar la "maldición de la dimensionalidad" creando demasiadas features
- Usar `pd.cut()` para bins equidistantes y `pd.qcut()` para bins por cuantiles

### ¿Para qué sirve?

El Feature Engineering sirve para:

- **Mejorar el rendimiento de modelos** proporcionando variables más informativas
- **Capturar relaciones no lineales** que los datos originales no expresan directamente
- **Enriquecer el análisis exploratorio** con métricas de negocio como márgenes y ratios
- **Simplificar patrones complejos** agrupando valores continuos en categorías interpretables
- **Extraer señales temporales** como estacionalidad, tendencias y efectos de día de la semana
- **Reducir dimensionalidad** combinando variables correlacionadas en una sola feature más expresiva

### ¿Cómo se usa?

En el código siguiente, creamos variables derivadas (margen, ingresos), aplicamos binning para categorizar precios, extraemos features temporales de fechas, generamos interacciones entre variables y aplicamos transformaciones matemáticas (logaritmo, raíz cuadrada).

In [ ]:
print("=" * 60)
print("FEATURE ENGINEERING BÁSICO")
print("=" * 60)

# Crear dataset
df_fe = pd.DataFrame(
    {
        "Producto": ["Laptop", "Mouse", "Teclado", "Monitor", "Webcam"],
        "Precio": [1200, 25, 75, 300, 80],
        "Costo": [800, 15, 45, 200, 50],
        "Ventas": [100, 500, 300, 150, 200],
        "Fecha": pd.date_range("2024-01-01", periods=5),
    }
)

print("\nDataset original:")
print(df_fe)

# 1. Crear variables derivadas
print("\n1. VARIABLES DERIVADAS")
df_fe["Margen"] = df_fe["Precio"] - df_fe["Costo"]
df_fe["Margen_Porcentaje"] = (df_fe["Margen"] / df_fe["Precio"]) * 100
df_fe["Ingresos"] = df_fe["Precio"] * df_fe["Ventas"]
df_fe["Ganancia"] = df_fe["Margen"] * df_fe["Ventas"]

print(df_fe[["Producto", "Precio", "Margen", "Margen_Porcentaje"]])

# 2. Binning (crear categorías)
print("\n2. BINNING (Categorización)")
df_fe["Precio_Categoria"] = pd.cut(
    df_fe["Precio"], bins=[0, 50, 200, 1500], labels=["Bajo", "Medio", "Alto"]
)
print(df_fe[["Producto", "Precio", "Precio_Categoria"]])

# 3. Features de fechas
print("\n3. FEATURES DE FECHAS")
df_fe["Mes"] = df_fe["Fecha"].dt.month
df_fe["Dia_Semana"] = df_fe["Fecha"].dt.dayofweek
df_fe["Es_Fin_Semana"] = df_fe["Dia_Semana"].isin([5, 6])
print(df_fe[["Fecha", "Mes", "Dia_Semana", "Es_Fin_Semana"]])

# 4. Interacciones
print("\n4. INTERACCIONES ENTRE VARIABLES")
df_fe["Precio_x_Ventas"] = df_fe["Precio"] * df_fe["Ventas"]
df_fe["Ratio_Precio_Costo"] = df_fe["Precio"] / df_fe["Costo"]
print(df_fe[["Producto", "Precio_x_Ventas", "Ratio_Precio_Costo"]])

# 5. Transformaciones
print("\n5. TRANSFORMACIONES MATEMÁTICAS")
df_fe["Precio_Log"] = np.log(df_fe["Precio"])
df_fe["Ventas_Sqrt"] = np.sqrt(df_fe["Ventas"])
print(df_fe[["Producto", "Precio", "Precio_Log", "Ventas", "Ventas_Sqrt"]])

print("\nDataFrame final con nuevas features:")
print("Columnas originales: 5")
print(f"Columnas después de FE: {len(df_fe.columns)}")
print(f"Nuevas features creadas: {len(df_fe.columns) - 5}")

## 9. Ejercicio Práctico Integrador: Pipeline de Limpieza Completo

### ¿Qué es?

Un **pipeline de limpieza** es la aplicación secuencial y sistemática de todas las técnicas de limpieza y preparación vistas en las secciones anteriores, formando un flujo completo que transforma un dataset "sucio" en uno listo para análisis o modelado.

**Pasos del pipeline integrador:**

| Paso | Técnica | Objetivo |
|---|---|---|
| 1 | Eliminación de duplicados | Remover registros repetidos |
| 2 | Validación de datos | Eliminar valores negativos o fuera de rango |
| 3 | Tratamiento de outliers | Remover valores extremos con IQR |
| 4 | Imputación de faltantes | Rellenar nulos con mediana (general y por grupo) |
| 5 | Feature Engineering | Crear nuevas variables a partir de existentes |
| 6 | Codificación categórica | Convertir variables textuales con One-Hot Encoding |
| 7 | Normalización | Estandarizar variables numéricas |

### ¿Para qué sirve?

El pipeline integrador sirve para:

- **Consolidar todas las técnicas** de limpieza en un flujo ordenado y reproducible
- **Practicar la secuencia completa** que se aplica en proyectos reales de datos
- **Entender el orden correcto** de los pasos (primero duplicados, luego validación, luego outliers, etc.)
- **Medir el impacto total** de la limpieza comparando el dataset inicial con el final
- **Crear una plantilla reutilizable** para futuros proyectos de limpieza de datos
- **Verificar la calidad final** del dataset con métricas de filas, columnas y valores faltantes

### ¿Cómo se usa?

En el código siguiente, creamos un dataset realista de 200 registros con múltiples problemas intencionales (duplicados, faltantes, outliers, valores negativos) y aplicamos los 7 pasos del pipeline en secuencia, generando un resumen comparativo del estado inicial y final del dataset.

In [ ]:
# Crear dataset realista con múltiples problemas
np.random.seed(42)
n = 200

df_sucio = pd.DataFrame(
    {
        "ID": list(range(1, n + 1)) + [50, 100],  # Duplicados
        "Nombre": ["Cliente_" + str(i) for i in range(1, n + 1)]
        + ["Cliente_50", "Cliente_100"],
        "Edad": np.random.randint(18, 80, n + 2),
        "Salario": np.random.randint(20000, 150000, n + 2),
        "Departamento": np.random.choice(["Ventas", "IT", "RRHH", "Marketing"], n + 2),
        "FechaIngreso": pd.date_range("2020-01-01", periods=n + 2, freq="D"),
    }
)

# Introducir problemas
# 1. Valores faltantes
df_sucio.loc[np.random.choice(df_sucio.index, 30), "Edad"] = np.nan
df_sucio.loc[np.random.choice(df_sucio.index, 25), "Salario"] = np.nan

# 2. Outliers
df_sucio.loc[5, "Salario"] = 500000
df_sucio.loc[10, "Edad"] = 150

# 3. Valores negativos
df_sucio.loc[15, "Salario"] = -10000

print("=" * 70)
print("PIPELINE DE LIMPIEZA COMPLETO".center(70))
print("=" * 70)

print("\n📊 ESTADO INICIAL DEL DATASET")
print("=" * 70)
print(f"Filas: {len(df_sucio)}")
print(f"Columnas: {len(df_sucio.columns)}")
print(f"\nValores faltantes:\n{df_sucio.isnull().sum()}")
print(f"\nDuplicados: {df_sucio.duplicated(subset=['ID']).sum()}")
print(f"\nPrimeras filas:\n{df_sucio.head()}")

# PASO 1: Eliminar duplicados
print("\n🔧 PASO 1: ELIMINAR DUPLICADOS")
filas_antes = len(df_sucio)
df_limpio = df_sucio.drop_duplicates(subset=["ID"], keep="first")
print(f"✓ Eliminados: {filas_antes - len(df_limpio)} registros")

# PASO 2: Validar y limpiar datos inválidos
print("\n🔧 PASO 2: VALIDAR Y LIMPIAR DATOS INVÁLIDOS")
invalidos_antes = len(df_limpio[(df_limpio["Salario"] < 0) | (df_limpio["Edad"] > 100)])
df_limpio = df_limpio[(df_limpio["Salario"] >= 0) | (df_limpio["Salario"].isna())]
df_limpio = df_limpio[(df_limpio["Edad"] <= 100) | (df_limpio["Edad"].isna())]
print(f"✓ Eliminados: {invalidos_antes} registros inválidos")

# PASO 3: Tratar outliers
print("\n🔧 PASO 3: TRATAR OUTLIERS (IQR)")
for col in ["Edad", "Salario"]:
    Q1 = df_limpio[col].quantile(0.25)
    Q3 = df_limpio[col].quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR

    outliers = df_limpio[(df_limpio[col] < limite_inf) | (df_limpio[col] > limite_sup)]
    print(f"  {col}: {len(outliers)} outliers detectados")

    # Remover outliers
    df_limpio = df_limpio[
        ((df_limpio[col] >= limite_inf) & (df_limpio[col] <= limite_sup))
        | df_limpio[col].isna()
    ]

# PASO 4: Imputar valores faltantes
print("\n🔧 PASO 4: IMPUTAR VALORES FALTANTES")
print(f"  Edad: {df_limpio['Edad'].isnull().sum()} faltantes -> Imputar con mediana")
df_limpio["Edad"] = df_limpio["Edad"].fillna(df_limpio["Edad"].median())

print(
    f"  Salario: {df_limpio['Salario'].isnull().sum()} faltantes -> Imputar con mediana por departamento"
)
df_limpio["Salario"] = df_limpio.groupby("Departamento")["Salario"].transform(
    lambda x: x.fillna(x.median())
)

# PASO 5: Feature Engineering
print("\n🔧 PASO 5: FEATURE ENGINEERING")
df_limpio["AñosEmpresa"] = (
    pd.Timestamp.now() - df_limpio["FechaIngreso"]
).dt.days / 365.25
df_limpio["RangoEdad"] = pd.cut(
    df_limpio["Edad"], bins=[0, 30, 50, 100], labels=["Joven", "Adulto", "Senior"]
)
df_limpio["RangoSalario"] = pd.qcut(
    df_limpio["Salario"], q=3, labels=["Bajo", "Medio", "Alto"]
)
print("✓ Nuevas features creadas: AñosEmpresa, RangoEdad, RangoSalario")

# PASO 6: Codificar variables categóricas
print("\n🔧 PASO 6: CODIFICAR VARIABLES CATEGÓRICAS")
df_limpio_encoded = pd.get_dummies(
    df_limpio, columns=["Departamento", "RangoEdad", "RangoSalario"], drop_first=True
)
print("✓ Variables codificadas con One-Hot Encoding")

# PASO 7: Normalizar variables numéricas
print("\n🔧 PASO 7: NORMALIZAR VARIABLES NUMÉRICAS")
scaler = StandardScaler()
df_limpio_encoded[["Edad_scaled", "Salario_scaled"]] = scaler.fit_transform(
    df_limpio_encoded[["Edad", "Salario"]]
)
print("✓ Edad y Salario estandarizados")

# RESUMEN FINAL
print("\n" + "=" * 70)
print("📈 RESUMEN FINAL")
print("=" * 70)
print(f"\nEstado Inicial:  {len(df_sucio)} filas, {len(df_sucio.columns)} columnas")
print(
    f"Estado Final:    {len(df_limpio_encoded)} filas, {len(df_limpio_encoded.columns)} columnas"
)
print(f"\nRegistros eliminados: {len(df_sucio) - len(df_limpio_encoded)}")
print(f"Features agregadas: {len(df_limpio_encoded.columns) - len(df_sucio.columns)}")
print(f"Valores faltantes restantes: {df_limpio_encoded.isnull().sum().sum()}")

print("\n✅ PIPELINE DE LIMPIEZA COMPLETADO")
print("\nPrimeras filas del dataset limpio:")
print(df_limpio_encoded.head())

# Exportar dataset limpio
df_limpio_encoded.to_csv("dataset_limpio.csv", index=False)
print("\n💾 Dataset limpio exportado a 'dataset_limpio.csv'")